In [3]:
import math
import pandas as pd
import requests


# 1. FETCH DATA
def fetch_openfda_data():
    base_url = "https://api.fda.gov/device/covid19serology.json"
    limit = 1000
    skip = 0
    records = []

    print("Fetching data from openFDA API...")
    while True:
        res = requests.get(f"{base_url}?limit={limit}&skip={skip}")

        if res.status_code == 404:
            break

        res.raise_for_status()
        batch = res.json().get("results", [])

        if not batch:
            break

        records.extend(batch)
        skip += limit

    print(f"Total raw records loaded: {len(records)}")
    return pd.DataFrame(records)


# 2. STATISTICAL HELPER: WILSON SCORE 95% CONFIDENCE INTERVAL
def calculate_wilson_ci(k, n, confidence_level=1.96):
    """Calculates Wilson Score 95% Confidence Interval for binomial proportions.

    k = successes (true negatives), n = total trials (negative samples)
    """
    if n == 0 or pd.isna(n):
        return 0.0, 0.0, 0.0

    p = k / n
    z = confidence_level  # 1.96 corresponds to 95% confidence

    denominator = 1 + (z**2 / n)
    center = p + (z**2 / (2 * n))
    spread = z * math.sqrt((p * (1 - p) / n) + (z**2 / (4 * n**2)))

    lower_bound = max(0.0, ((center - spread) / denominator) * 100)
    upper_bound = min(100.0, ((center + spread) / denominator) * 100)
    point_estimate = p * 100

    return point_estimate, lower_bound, upper_bound


# 3. CLEAN & PREPARE DATA
def clean_data(df):
    df = df.copy()

    # Handle missing string values
    df = df.replace(["NA", ""], None)

    # Convert numeric titers
    titer_cols = [col for col in df.columns if "titer" in col]
    for col in titer_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Helper boolean indicators
    df["is_true_negative"] = df["antibody_agree"] == "TN"
    df["is_false_positive"] = df["antibody_agree"] == "FP"
    df["is_valid_control"] = df["control"] == "Pass"

    return df


# 4. RIGOROUS DEVICE PERFORMANCE ANALYSIS
def analyze_device_specificity(df, target_threshold=95.0):
    # Filter: True negative ground-truth AND valid kit controls
    valid_negatives = df[
        (df["antibody_truth"] == "Negative") & (df["is_valid_control"])
    ]

    # Aggregation
    stats = (
        valid_negatives.groupby("device")
        .agg(
            n_evaluations=("sample_id", "count"),
            true_negatives=("is_true_negative", "sum"),
        )
        .reset_index()
    )

    # Compute Wilson Score Confidence Intervals
    ci_results = stats.apply(
        lambda row: calculate_wilson_ci(
            row["true_negatives"], row["n_evaluations"]
        ),
        axis=1,
    )

    stats["specificity_pct"] = [res[0] for res in ci_results]
    stats["ci_lower_95"] = [res[1] for res in ci_results]
    stats["ci_upper_95"] = [res[2] for res in ci_results]

    # Identify devices failing FDA benchmark (Point estimate < threshold OR Lower Bound < 90%)
    failed = stats[stats["specificity_pct"] < target_threshold].sort_values(
        by="specificity_pct"
    )

    return failed


# 5. CROSS-REACTIVITY ANALYSIS (NON-COVID COHORTS ONLY)
def analyze_cross_reactivity(df):
    # Cross-reactivity evaluates non-COVID negative samples (e.g., HIV+, ANA+)
    cross_reactivity_cohorts = df[
        (df["antibody_truth"] == "Negative")
        & (df["is_valid_control"])
        & (df["group"] != "Negative")  # Exclude generic healthy negatives
    ]

    group_stats = (
        cross_reactivity_cohorts.groupby("group")
        .agg(
            total_samples=("sample_id", "count"),
            false_positives=("is_false_positive", "sum"),
            avg_igg_titer=("igg_titer", "mean"),
        )
        .reset_index()
    )

    group_stats["false_positive_rate"] = (
        group_stats["false_positives"] / group_stats["total_samples"]
    ) * 100

    return group_stats.sort_values(by="false_positive_rate", ascending=False)


# --- EXECUTION ---
if __name__ == "__main__":
    raw_df = fetch_openfda_data()
    df = clean_data(raw_df)

    # Overall Dataset Specificity (Filtered for Valid Controls)
    valid_negs = df[
        (df["antibody_truth"] == "Negative") & (df["is_valid_control"])
    ]
    tn_count = valid_negs["is_true_negative"].sum()
    n_count = len(valid_negs)

    point_est, lower_ci, upper_ci = calculate_wilson_ci(tn_count, n_count)

    print("\n" + "=" * 65)
    print("OVERALL DATASET SPECIFICITY (VALID CONTROLS ONLY)")
    print(
        f"Point Estimate: {point_est:.2f}% | 95% CI: [{lower_ci:.2f}% - {upper_ci:.2f}%]"
    )
    print(f"Sample Size: {tn_count}/{n_count} True Negatives")
    print("=" * 65)

    # Output: Underperforming Devices with 95% CI
    print("\n--- DEVICES FAILING 95% SPECIFICITY BENCHMARK ---")
    failed_devices = analyze_device_specificity(df, target_threshold=95.0)

    print(
        failed_devices.head(10).to_string(
            index=False,
            formatters={
                "specificity_pct": "{:.2f}%".format,
                "ci_lower_95": "{:.2f}%".format,
                "ci_upper_95": "{:.2f}%".format,
            },
        )
    )

    # Output: Cross-Reactivity Risk
    print("\n--- DISEASE-SPECIFIC CROSS-REACTIVITY (FALSE POSITIVE RATES) ---")
    cross_reactivity = analyze_cross_reactivity(df)

    print(
        cross_reactivity.to_string(
            index=False,
            formatters={
                "avg_igg_titer": "{:.2f}".format,
                "false_positive_rate": "{:.2f}%".format,
            },
        )
    )

Fetching data from openFDA API...
Total raw records loaded: 13420

OVERALL DATASET SPECIFICITY (VALID CONTROLS ONLY)
Point Estimate: 94.79% | 95% CI: [94.33% - 95.22%]
Sample Size: 9249/9757 True Negatives

--- DEVICES FAILING 95% SPECIFICITY BENCHMARK ---
                                                                                                 device  n_evaluations  true_negatives specificity_pct ci_lower_95 ci_upper_95
Novel Coronavirus (SARS-CoV-2) IgM and IgG Dual Combined Antibody Detection Kit (Colloidal Gold Method)             80              46          57.50%      46.57%      67.74%
                                                              Rapid C2T Total Antibodies (IgG/IgM) Card             80              48          60.00%      49.05%      70.04%
                                                                SARS-COV-2 IgM/IgG Combo Rapid Test Kit             80              50          62.50%      51.55%      72.31%
                                          V